# 🔬 Advanced Analytics & Risk Metrics — Bluestock MF Capstone
## Day 6: VaR, Rolling Sharpe, Cohort Analysis, Recommender, HHI

**Tools:** Python, Pandas, NumPy, Scipy, Matplotlib, Seaborn  
**Risk-Free Rate:** 6.5% (RBI Repo Rate) | **Trading Days:** 252

---


## ⚙️ Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import Image, display

BASE   = Path().resolve().parent
RAW    = BASE / "data" / "raw"
PROC   = BASE / "data" / "processed"
CHARTS = BASE / "reports" / "charts"

RF_DAILY     = 0.065 / 252
TRADING_DAYS = 252

COLORS = ["#6c8ebf","#82b366","#d6a91a","#e07b54","#9b72cf",
          "#4db6ac","#e57373","#81c784","#ffb74d","#64b5f6"]

plt.rcParams.update({
    "figure.facecolor":"#0f1117","axes.facecolor":"#1a1d2e",
    "axes.labelcolor":"#ccccdd","xtick.color":"#aaaacc",
    "ytick.color":"#aaaacc","text.color":"#ccccdd",
    "grid.color":"#2a2d3e","grid.alpha":0.5,
})

nav      = pd.read_csv(RAW/"02_nav_history.csv", parse_dates=["date"])
funds    = pd.read_csv(RAW/"01_fund_master.csv")
txn      = pd.read_csv(RAW/"08_investor_transactions.csv", parse_dates=["transaction_date"])
holdings = pd.read_csv(RAW/"09_portfolio_holdings.csv")
perf     = pd.read_csv(RAW/"07_scheme_performance.csv")

nav = nav.merge(funds[["amfi_code","scheme_name","fund_house","sub_category","risk_category"]], on="amfi_code", how="left")
nav = nav.sort_values(["amfi_code","date"]).reset_index(drop=True)
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()

print("✅ All data loaded!")

---
## 📉 Task 1: Historical VaR (95%) & CVaR
**VaR 95%** = 5th percentile of daily return distribution  
**CVaR 95%** = Mean of all returns below VaR threshold (Expected Shortfall)


In [ ]:
var_results = []
for code, grp in nav.groupby("amfi_code"):
    ret = grp["daily_return"].dropna()
    if len(ret) < 50: continue
    var_95  = np.percentile(ret, 5)
    cvar_95 = ret[ret <= var_95].mean()
    var_results.append({
        "amfi_code"     : code,
        "scheme_name"   : grp["scheme_name"].iloc[0],
        "fund_house"    : grp["fund_house"].iloc[0],
        "sub_category"  : grp["sub_category"].iloc[0],
        "risk_category" : grp["risk_category"].iloc[0],
        "var_95_pct"    : round(var_95 * 100, 4),
        "cvar_95_pct"   : round(cvar_95 * 100, 4),
        "var_95_ann_pct": round(var_95 * np.sqrt(252) * 100, 4),
    })

var_df = pd.DataFrame(var_results).sort_values("var_95_pct")
print(f"VaR computed for {len(var_df)} funds")
print(f"\nRiskiest fund : {var_df.iloc[0]['scheme_name'][:45]}")
print(f"  VaR  : {var_df.iloc[0]['var_95_pct']:.4f}%")
print(f"  CVaR : {var_df.iloc[0]['cvar_95_pct']:.4f}%")
print(f"\nSafest fund   : {var_df.iloc[-1]['scheme_name'][:45]}")
print(f"  VaR  : {var_df.iloc[-1]['var_95_pct']:.4f}%")
print(f"  CVaR : {var_df.iloc[-1]['cvar_95_pct']:.4f}%")
var_df.head(10)

In [ ]:
# VaR chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor("#0f1117")

# Bar chart — VaR by fund
top10_var = var_df.head(10)
short_names = [n.split("-")[0].strip()[:20] for n in top10_var["scheme_name"]]
axes[0].barh(short_names, top10_var["var_95_pct"], color=COLORS[3], alpha=0.85)
axes[0].set_title("Top 10 Riskiest Funds by VaR 95%")
axes[0].set_xlabel("VaR 95% (Daily %)")
axes[0].grid(True, axis="x")

# VaR vs CVaR scatter
axes[1].scatter(var_df["var_95_pct"], var_df["cvar_95_pct"],
                c=[COLORS[i%len(COLORS)] for i in range(len(var_df))], s=80, alpha=0.85)
axes[1].set_title("VaR vs CVaR — All 40 Funds")
axes[1].set_xlabel("VaR 95% (%)"); axes[1].set_ylabel("CVaR 95% (%)")
axes[1].grid(True)

plt.suptitle("Value at Risk Analysis", fontsize=14)
plt.tight_layout(); plt.show()

var_df.to_csv(PROC/"var_cvar_report.csv", index=False)
print("✅ var_cvar_report.csv saved!")

---
## 📈 Task 2: Rolling 90-Day Sharpe Ratio
**Formula:** `rolling_sharpe = returns.rolling(90).mean() / returns.rolling(90).std() × √252`


In [ ]:
display(Image(str(CHARTS/"D6_rolling_sharpe.png"), width=900))

In [ ]:
# Interactive rolling Sharpe
top5 = perf.sort_values("sharpe_ratio", ascending=False).head(5)["amfi_code"].tolist()

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor("#0f1117")

for i, code in enumerate(top5):
    grp = nav[nav["amfi_code"]==code].sort_values("date").copy()
    ret = grp.set_index("date")["daily_return"]
    rs  = (ret.rolling(90).mean() - RF_DAILY) / ret.rolling(90).std() * np.sqrt(252)
    name = grp["scheme_name"].iloc[0].split("-")[0].strip()[:20]
    ax.plot(rs.index, rs.values, color=COLORS[i], linewidth=1.5, label=name)

ax.axhline(0, color="#ffb74d", linestyle="--", linewidth=1, label="Sharpe=0")
ax.axhline(1, color="#82b366", linestyle=":", linewidth=1, label="Sharpe=1")
ax.set_title("Rolling 90-Day Sharpe Ratio — Top 5 Funds")
ax.set_xlabel("Date"); ax.set_ylabel("Rolling Sharpe")
ax.legend(fontsize=8, framealpha=0.3); ax.grid(True)
plt.tight_layout(); plt.show()

---
## 👥 Task 3: Investor Cohort Analysis
**Groups investors by first transaction year. Computes avg SIP, total invested, top fund.**


In [ ]:
txn["year"] = txn["transaction_date"].dt.year
first_yr = txn.groupby("investor_id")["year"].min().reset_index()
first_yr.columns = ["investor_id","cohort_year"]
txn_c = txn.merge(first_yr, on="investor_id", how="left")

cohort = (txn_c[txn_c["transaction_type"]=="SIP"]
    .groupby("cohort_year")
    .agg(
        investor_count   =("investor_id","nunique"),
        avg_sip_amount   =("amount_inr","mean"),
        total_invested   =("amount_inr","sum"),
        transaction_count=("investor_id","count"),
    ).round(2).reset_index())

top_fund = (txn_c.groupby(["cohort_year","amfi_code"])["amount_inr"]
    .sum().reset_index()
    .sort_values("amount_inr",ascending=False)
    .drop_duplicates("cohort_year")
    .merge(funds[["amfi_code","scheme_name"]],on="amfi_code",how="left")
    [["cohort_year","scheme_name"]].rename(columns={"scheme_name":"top_fund"}))

cohort = cohort.merge(top_fund, on="cohort_year", how="left")
cohort.to_csv(PROC/"cohort_analysis.csv", index=False)
print("Cohort Analysis:")
cohort

---
## ⏱️ Task 4: SIP Continuity Analysis
**Investors with 6+ SIPs. Flag those with avg gap > 35 days as "at-risk".**


In [ ]:
sip_txn = txn[txn["transaction_type"]=="SIP"].sort_values(["investor_id","transaction_date"])
continuity = []
for inv_id, grp in sip_txn.groupby("investor_id"):
    if len(grp) < 6: continue
    gaps = grp["transaction_date"].diff().dt.days.dropna()
    avg_gap = gaps.mean()
    continuity.append({
        "investor_id"  : inv_id,
        "sip_count"    : len(grp),
        "avg_gap_days" : round(avg_gap, 1),
        "total_invested": grp["amount_inr"].sum(),
        "at_risk"      : "Yes" if avg_gap > 35 else "No",
        "state"        : grp["state"].iloc[0],
        "age_group"    : grp["age_group"].iloc[0],
    })

cont_df = pd.DataFrame(continuity)
at_risk = cont_df[cont_df["at_risk"]=="Yes"]
print(f"Total investors (6+ SIPs) : {len(cont_df):,}")
print(f"At-risk investors         : {len(at_risk):,} ({len(at_risk)/len(cont_df)*100:.1f}%)")
print(f"Regular investors         : {len(cont_df)-len(at_risk):,}")

fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.patch.set_facecolor("#0f1117")
axes[0].hist(cont_df["avg_gap_days"], bins=30, color=COLORS[0], alpha=0.85, edgecolor="#1a1d2e")
axes[0].axvline(35, color="#e07b54", linestyle="--", linewidth=2, label="35-day threshold")
axes[0].set_title("SIP Gap Distribution"); axes[0].set_xlabel("Avg Gap (days)")
axes[0].legend()

cont_df["at_risk"].value_counts().plot(kind="bar", ax=axes[1], color=[COLORS[1],COLORS[3]], alpha=0.85)
axes[1].set_title("At-Risk vs Regular SIP Investors"); axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout(); plt.show()
cont_df.to_csv(PROC/"sip_continuity.csv", index=False)
print("✅ sip_continuity.csv saved!")

---
## 🤖 Task 5: Fund Recommender
**Input:** Risk appetite → **Output:** Top 3 funds by Sharpe ratio


In [ ]:
def recommend_funds(risk_appetite):
    merged = perf.merge(funds[["amfi_code","risk_category","expense_ratio_pct",
                                "sub_category","plan"]], on="amfi_code", how="left")
    risk_map = {
        "Low"            : ["Low"],
        "Moderate"       : ["Moderate","Low"],
        "Moderately High": ["Moderately High","Moderate"],
        "High"           : ["High","Moderately High"],
        "Very High"      : ["Very High","High"],
    }
    grades   = risk_map.get(risk_appetite, [risk_appetite])
    filtered = merged[merged["risk_grade"].isin(grades)]
    top3 = (filtered.sort_values("sharpe_ratio", ascending=False)
                    .head(3)[["scheme_name","fund_house","sub_category",
                               "risk_grade","sharpe_ratio","return_3yr_pct",
                               "expense_ratio_pct","morningstar_rating"]]
                    .reset_index(drop=True))
    top3.index += 1
    return top3

for risk in ["Low","Moderate","Moderately High","High","Very High"]:
    print(f"\n{'='*60}")
    print(f"  Risk Appetite: {risk}")
    print('='*60)
    display(recommend_funds(risk))

---
## 📊 Task 6: Sector HHI Concentration
**HHI = Σ(weight_i²) — Higher HHI = More concentrated portfolio**


In [ ]:
equity_codes = funds[funds["category"]=="Equity"]["amfi_code"].tolist()
hhi_results = []
for code, grp in holdings[holdings["amfi_code"].isin(equity_codes)].groupby("amfi_code"):
    weights = grp["weight_pct"] / 100
    hhi = (weights**2).sum()
    name = funds[funds["amfi_code"]==code]["scheme_name"].values
    hhi_results.append({
        "amfi_code"    : code,
        "scheme_name"  : name[0] if len(name)>0 else "Unknown",
        "hhi_score"    : round(hhi, 4),
        "concentration": "High" if hhi > 0.15 else "Moderate" if hhi > 0.08 else "Low",
        "num_holdings" : len(grp),
    })

hhi_df = pd.DataFrame(hhi_results).sort_values("hhi_score", ascending=False)

fig, ax = plt.subplots(figsize=(12,7))
fig.patch.set_facecolor("#0f1117")
colors_hhi = ["#e07b54" if h=="High" else "#d6a91a" if h=="Moderate" else "#82b366"
              for h in hhi_df["concentration"]]
short = [n.split("-")[0].strip()[:22] for n in hhi_df["scheme_name"]]
ax.barh(short, hhi_df["hhi_score"], color=colors_hhi, alpha=0.85)
ax.axvline(0.15, color="#e07b54", linestyle="--", linewidth=1.5, label="High HHI (>0.15)")
ax.axvline(0.08, color="#d6a91a", linestyle=":", linewidth=1.5, label="Moderate HHI (>0.08)")
ax.set_title("Sector HHI Concentration — Equity Funds"); ax.set_xlabel("HHI Score")
ax.legend(framealpha=0.3); ax.grid(True, axis="x")
plt.tight_layout(); plt.show()

hhi_df.to_csv(PROC/"sector_hhi.csv", index=False)
print("✅ sector_hhi.csv saved!")
hhi_df

---
## 📝 5 Advanced Insights

| # | Insight |
|---|---------|
| 1 | **Highest VaR:** SBI Small Cap Fund has the highest daily VaR of -2.69%, meaning on worst 5% days investors can lose 2.69% — typical for Small Cap funds |
| 2 | **Safest fund:** ICICI Pru Liquid Fund has VaR of only -0.02%, confirming Liquid funds are near risk-free investments |
| 3 | **Investor cohorts:** 2024 cohort dominates with 4,624 investors avg SIP ₹10,997 — showing surge in new retail investors post-COVID |
| 4 | **SIP continuity:** 97.8% of investors with 6+ SIPs are "at-risk" (gap > 35 days), suggesting many investors miss monthly SIP dates |
| 5 | **Portfolio concentration:** Axis Bluechip Fund has highest HHI (0.2064) — most concentrated portfolio. SBI Small Cap is most diversified (0.1073) |
